# Checkpoint III · Examen blanc — notebook de l'examen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_3/02_examen_notebook.ipynb)

Ce notebook accompagne le sujet `01_examen_sujet.md`.

- **Partie A, pendant l'examen** : les deux questions de code, CP3.6 (🐛, 2 points, avec scikit-learn) et CP3.7 (🔨, 2 points, avec NumPy seulement). Écris ton code dans les cellules à compléter, sans `mylearn`, sans les corrigés et sans assistant IA ; `help()` et la documentation officielle sont permis. Rien n'est vérifié pendant l'examen : les cellules « essai » montrent seulement ce que fait ton code. Tes réponses chiffrées et tes explications vont sur ta feuille (`04_mes_reponses.md`).
- **Partie B, après l'examen** : la vérification automatique de ton code et des réponses de ta feuille. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`.

« Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳ (partie A) ou ⏸️ (partie B).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_3/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP3`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Partie A · Pendant l'examen : les deux questions de code

In [ ]:
# Data and tools of the two code questions (run this cell first)
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

# CP3.6: 4 000 districts of California (1990 census)
california = wb.datasets.load_california()
FEATURES = california.columns[:8].tolist()                    # MedInc, HouseAge, AveRooms, AveBedrms, Population, ...
rows = np.sort(np.random.default_rng(2606).choice(len(california), 4000, replace=False))
X = california[FEATURES].to_numpy(dtype=float)[rows]          # shape (4000, 8)
y = california["MedHouseVal"].to_numpy(dtype=float)[rows]     # median house value (hundreds of thousands of $)
K_GRID = [1, 3, 5, 10, 20, 40]                                # numbers of neighbours tried


# CP3.7: the probabilities of a binary classifier, for 600 validation and 600 test examples
def make_scores(n, seed):
    """n labels (about 25 % of positives) and their probability of being positive (calibrated, 3 decimals)."""
    rng = np.random.default_rng(seed)
    labels = (rng.random(n) < 0.25).astype(int)
    score = rng.normal(np.where(labels == 1, 1.0, -1.0), 1.0)
    return np.round(1 / (1 + 3 * np.exp(-2 * score)), 3), labels


PROBA_VAL, Y_VAL = make_scores(600, 3707)
PROBA_TEST, Y_TEST = make_scores(600, 3708)
THRESHOLDS = np.round(np.arange(1, 100) / 100, 2)             # 0.01, 0.02, ..., 0.99
COST_FN, COST_FP = 5, 1                                       # a false negative costs 5, a false positive 1
_ORIGINALS = {"X": X.copy(), "y": y.copy(), "PROBA_VAL": PROBA_VAL.copy(), "Y_VAL": Y_VAL.copy(),
              "PROBA_TEST": PROBA_TEST.copy(), "Y_TEST": Y_TEST.copy()}   # part B restores them if a try changed them
print("X:", X.shape, "· y:", y.shape, "· validation:", PROBA_VAL.shape, "· positives:", int(Y_VAL.sum()))

### CP3.6 — Trouver trois fuites dans un script d'entraînement 🐛 ★★ ⏱️ 12 min · 2 points
**Objectif :** repérer ce qu'une évaluation a appris de son jeu de test, puis refaire l'évaluation sans fuite.

Un collègue prédit la valeur médiane des logements des 4 000 districts de `X` et `y` avec un kNN de régression, dont il choisit le nombre de voisins dans `K_GRID`. Sa fonction `colleague_report(X, y)`, dont les lignes sont marquées (A) à (F), annonce au client un $R^2$ de test de 0,93. Exécute-la (cellule suivante).

a) Trois de ses lignes contiennent une fuite de données : leurs lettres (sur ta feuille). **[0,6]**
b) Laquelle de ces fuites fausse vraiment son score, et pourquoi ? Pourquoi les deux autres ne le changent-elles presque pas ici, et pourquoi faut-il quand même les corriger ? (sur ta feuille) **[0,4]**
c) Écris `honest_report(X, y)`, qui fait le même travail **sans fuite** et renvoie `(best_k, test_r2)` : garde la transformation de la ligne (B) telle quelle (la valeur attendue à la question d est calculée avec elle) et retire les fuites ; découpe d'abord, avec le même `train_test_split(..., test_size=0.25, random_state=0)` ; choisis le nombre de voisins par une `GridSearchCV` à 5 folds (`cv=5`, score $R^2$ par défaut) sur l'entraînement seul, avec un `Pipeline` qui standardise puis applique le kNN ; mesure enfin le $R^2$ de test, une seule fois. La fonction ne modifie pas `X`. **[0,6]**
d) Les valeurs que renvoie ta fonction : $[\text{best\_k}, R^2_{\text{test}}]$ (3 décimales, sur ta feuille). **[0,2]**
e) Que dis-tu au client ? (sur ta feuille) **[0,2]**

In [ ]:
# The colleague's code (do not change this cell: run it and read what it prints)
def colleague_report(X, y):
    X = np.column_stack([X, y / X[:, 2]])                                        # (A) a 9th feature: the value per room
    X[:, 4] = np.log1p(X[:, 4])                                                  # (B) the log of the population, row by row
    X = StandardScaler().fit_transform(X)                                        # (C) standardize the 9 columns
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)   # (D) split
    scores = {k: KNeighborsRegressor(n_neighbors=k).fit(X_train, y_train).score(X_test, y_test)
              for k in K_GRID}                                                   # (E) the R² of each k on the test
    best = max(scores, key=scores.get)
    return best, scores[best]                                                    # (F) the k kept and its test R²


best_colleague, r2_colleague = colleague_report(X, y)
print(f"the colleague keeps k = {best_colleague}, with a test R² of {r2_colleague:.3f}")

In [ ]:
def honest_report(X, y):
    """The same evaluation without leakage: returns (best_k, test_r2), see CP3.6 c)."""
    raise NotImplementedError("honest_report")  # TODO CP3.6 c)

In [ ]:
# Try your function (no check during the exam: compare with what you expect)
with wb.attempt("CP3.6"):
    print("honest report:", honest_report(X, y))

### CP3.7 — Coder le seuil de décision de coût minimal et le comparer au seuil théorique 🔨 ★★ ⏱️ 14 min · 2 points
**Objectif :** choisir un seuil de décision d'après le coût des erreurs, sur la validation, et le confronter au calcul théorique.

Avec NumPy seulement. `PROBA_VAL` donne, pour 600 exemples de validation, la probabilité d'être positif qu'annonce un classifieur, et `Y_VAL` leurs labels (0 ou 1) ; `PROBA_TEST` et `Y_TEST`, la même chose pour 600 exemples de test. Un faux négatif coûte `COST_FN = 5`, un faux positif `COST_FP = 1`. La règle de décision : classe 1 quand la probabilité est **supérieure ou égale** au seuil $t$.

a) Écris `total_cost(y_true, y_pred, cost_fn, cost_fp)`, qui renvoie le coût total (un `int`) de décisions 0/1. **[0,3]**
b) Écris `min_cost_threshold(proba, y_true, cost_fn, cost_fp, thresholds)`, qui renvoie le couple `(t, coût)` du seuil de plus petit coût parmi `thresholds` (le premier dans l'ordre de `thresholds` en cas d'égalité). Les deux fonctions ne modifient pas leurs arguments. **[0,5]**
c) Avec `THRESHOLDS` (0,01 ; 0,02 ; … ; 0,99), le seuil et le coût que donne ta fonction sur la validation : $[t, \text{coût}]$ (sur ta feuille). **[0,3]**
d) Le seuil théorique de coût moyen minimal, si les probabilités sont calibrées (4 décimales, sur ta feuille). Montre-le en comparant le coût moyen des deux décisions pour un exemple de probabilité $p$. **[0,3]**
e) Les coûts sur le test, avec le seuil de c), puis avec le seuil 0,5 : $[\text{coût}(t), \text{coût}(0{,}5)]$ (sur ta feuille). **[0,3]**
f) Pourquoi c) et d) ne donnent-ils pas le même seuil ? Lequel déploierais-tu, et pourquoi ? (sur ta feuille) **[0,3]**

In [ ]:
def total_cost(y_true, y_pred, cost_fn, cost_fp):
    """Total cost (an int) of the 0/1 decisions y_pred, see CP3.7 a)."""
    raise NotImplementedError("total_cost")  # TODO CP3.7 a)


def min_cost_threshold(proba, y_true, cost_fn, cost_fp, thresholds):
    """(t, cost) of the threshold of smallest total cost among thresholds (the first one if tied), CP3.7 b)."""
    raise NotImplementedError("min_cost_threshold")  # TODO CP3.7 b)

In [ ]:
# Try your functions (no check during the exam)
with wb.attempt("CP3.7"):
    print("cost of [1, 0, 1] for the labels [1, 1, 0]:", total_cost(np.array([1, 1, 0]), np.array([1, 0, 1]), 5, 1))
with wb.attempt("CP3.7"):
    chosen = min_cost_threshold(PROBA_VAL, Y_VAL, COST_FN, COST_FP, THRESHOLDS)
    print("validation: (t, cost) =", chosen)

---

### 🛑 Fin de la partie A

L'examen s'arrête ici. Rends ta copie (enregistre ce notebook et ta feuille), puis, seulement après, passe à la partie B.

## Partie B · Après l'examen : vérification automatique

**Ne commence cette partie qu'une fois ta copie terminée**, examen rendu. Elle vérifie le code de la partie A, puis les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Elle ne note pas : les démarches, les justifications et les questions rédigées se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Exécute la vérification du code (CP3.6 et CP3.7).
3. Reporte ensuite chaque réponse de ta feuille : **la valeur** que tu as écrite, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). En Python, le séparateur décimal est un **point** (`0.125`), et les lettres s'écrivent entre guillemets (`"XYZ"`).

Si tu as fermé le notebook depuis l'examen, exécute d'abord la cellule de configuration (tout en haut) et les cellules de la partie A, sauf les essais.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells of part B

import re

RESULTS = {}        # sub-ID -> result of wb.check
PROPERTIES = {}     # question -> [(property, passed)], reset each time its check cell runs
_REMINDED = []      # the full reminder is printed once, then a short one


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans la première cellule de la partie B.")
        _REMINDED.append(True)
        return False
    if "wb" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True


def ready(question, *names):
    """True when the cells of part A that define `names` have run (after a kernel restart, run them again)."""
    missing = [name for name in names if name not in globals()]
    if missing:
        verb = "n'existe" if len(missing) == 1 else "n'existent"
        print(f"⚠️ {question} : exécute d'abord les cellules de la partie A ({', '.join(missing)} {verb} pas encore).")
    return not missing


def verdict(question, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message, and remember the result."""
    PROPERTIES.setdefault(question, []).append((success or failure, bool(ok)))
    print(f"✅ {question} : {success}" if ok else f"❌ {question} : {failure}")
    return bool(ok)


_LIBRARY_ADVICE = " ; si elle fait partie de `mylearn`, ses tests (`pytest`) disent quel cas pose problème."


def code_check(ex_id, value, reminder=""):
    """wb.check of a value computed by your exam function (no advice about mylearn: it is not used here)."""
    result = wb.check(ex_id, value, computed=True, quiet=True)
    message = result.message.replace(_LIBRARY_ADVICE, ".")
    if not result and reminder and not message.startswith("Erreur classique"):
        message += f" Rappel : {reminder}"
    print(f"{'✅' if result else '❌'} Ex {ex_id} : {message}")
    RESULTS[ex_id] = result
    return result


def as_letters(value):
    """Letters typed as "ACE", "A, C et E" or ["A", "C", "E"] -> "ACE" (`...` stays `...`)."""
    if value is ... or value is None:
        return value
    if isinstance(value, (list, tuple)):
        value = " ".join(str(v) for v in value)
    value = re.sub(r"(?i)\b(et|and|ou|or)\b", " ", str(value))
    return "".join(ch for ch in value if ch.isalpha()).upper()


def as_curve(value):
    """A curve number typed as 2, "2", "(2)" or "②" -> 2 (anything else is left as typed)."""
    if isinstance(value, str):
        text = value.strip().replace("①", "1").replace("②", "2").strip("()[] .")
        if text.isdigit():
            return int(text)
    return value


def restore_data(question, names):
    """Put back the data of part A if a try of part A changed them in place (a function that writes into its input)."""
    changed = [name for name in names
               if name not in globals() or not np.array_equal(globals()[name], _ORIGINALS[name])]
    for name in changed:
        globals()[name] = _ORIGINALS[name].copy()
    if changed:
        print(f"⚠️ {question} : {', '.join(changed)} avai{'en' if len(changed) > 1 else ''}t changé depuis la cellule des "
              f"données (une fonction de la partie A écrit-elle dans ce qu'elle reçoit ?) : la vérification repart "
              f"des données d'origine.")

### Le code de la partie A

In [ ]:
# Checks of honest_report (CP3.6)
if exam_over() and ready("CP3.6", "np", "X", "y", "K_GRID", "colleague_report", "honest_report", "_ORIGINALS",
                         "california"):
    PROPERTIES["CP3.6"] = []
    other_rows = np.sort(np.random.default_rng(2710).choice(len(california), 4000, replace=False))
    X_OTHER_6 = california[FEATURES].to_numpy(dtype=float)[other_rows]       # 4 000 other districts
    y_OTHER_6 = california["MedHouseVal"].to_numpy(dtype=float)[other_rows]
    OTHER_6, OTHER_6_ON_TEST = (5, 0.6544), 10      # the honest (best_k, R²) there, and the k that the test prefers
    RESULTS.pop("CP3.6d", None)
    restore_data("CP3.6", ["X", "y"])
    X_check, y_check = X.copy(), y.copy()
    report, ran = None, False
    try:
        report = honest_report(X_check, y_check)
        ran = True
    except NotImplementedError:
        print("⏳ CP3.6 : honest_report n'est pas encore écrite.")
    except Exception as exc:                   # a bug of the exam code must not stop part B
        verdict("CP3.6", False, "", f"honest_report(X, y) lève une erreur : {type(exc).__name__}: {exc}")
    if ran and report is None:
        verdict("CP3.6", False, "", "honest_report ne renvoie rien : as-tu oublié le return ?")
    if report is not None:
        verdict("CP3.6", np.array_equal(X_check, X) and np.array_equal(y_check, y), "X et y ne sont pas modifiés.",
                "ta fonction modifie X (ou y) : la ligne (B) écrit dans le tableau qu'elle reçoit ; travaille sur une "
                "copie (X = X.copy()) avant de transformer la colonne de la population.")
        pair = None
        try:
            pair = [float(v) for v in report]
        except (TypeError, ValueError):
            pass
        if pair is None or len(pair) != 2:
            verdict("CP3.6", False, "", f"honest_report doit renvoyer un couple (best_k, test_r2), pas "
                    f"{type(report).__name__} {report!r}"[:300])
        else:
            verdict("CP3.6", pair[0] in K_GRID, "le nombre de voisins retenu fait partie de K_GRID.",
                    f"best_k vaut {pair[0]:g} : ce doit être l'une des valeurs de K_GRID, lue dans best_params_.")
            leaky = colleague_report(X, y)[1]
            verdict("CP3.6", pair[1] < leaky - 0.05,
                    "sans la fuite de la ligne (A), le R² de test est nettement plus bas que celui du collègue.",
                    "ton R² reste près de celui du collègue : une feature calculée à partir de y reste-t-elle dans X ?")
            try:                               # other districts, where the k of the test is not the k of the folds
                other = [float(v) for v in honest_report(X_OTHER_6.copy(), y_OTHER_6.copy())]
            except Exception as exc:
                other = f"{type(exc).__name__}: {exc}"
            verdict("CP3.6", isinstance(other, list) and len(other) == 2 and other[0] == OTHER_6[0]
                    and abs(other[1] - OTHER_6[1]) < 5e-4,
                    "sur d'autres districts, k est choisi par la validation croisée, et le test ne sert qu'à la fin.",
                    f"sur 4 000 autres districts, la validation croisée choisit k = {OTHER_6[0]} (R² du test "
                    f"{OTHER_6[1]:.4f}), et ta fonction renvoie {other!r} : "
                    + ("k est choisi d'après le score du TEST (la fuite (E)), ou bien le StandardScaler est appris une "
                       "fois sur tout l'entraînement, hors du pipeline, et la validation croisée ne le réapprend pas "
                       "dans chaque fold, ou bien tes folds ne sont pas ceux de cv=5 (sans mélange) que demande "
                       "l'énoncé." if isinstance(other, list) and len(other) == 2 and other[0] == OTHER_6_ON_TEST else
                       "relis l'ordre : découper, chercher k par validation croisée sur l'entraînement, puis évaluer."))
            code_check("CP3.6d", pair, "découpe d'abord ; un Pipeline (StandardScaler, kNN) dans une GridSearchCV à "
                       "5 folds sur l'entraînement ; puis le R² du test, une seule fois.")

In [ ]:
# Checks of total_cost and min_cost_threshold (CP3.7)
if exam_over() and ready("CP3.7", "np", "PROBA_VAL", "Y_VAL", "PROBA_TEST", "Y_TEST", "THRESHOLDS", "total_cost",
                         "min_cost_threshold"):
    PROPERTIES["CP3.7"] = []
    for sub in ("CP3.7c", "CP3.7e"):
        RESULTS.pop(sub, None)
    restore_data("CP3.7", ["PROBA_VAL", "Y_VAL", "PROBA_TEST", "Y_TEST"])

    def safe(function, *args):
        """function(*args), or the text of the error it raises (a bug of the exam code must not stop part B)."""
        try:
            return function(*args)
        except Exception as exc:
            return f"{type(exc).__name__}: {exc}"

    def as_pair(value):
        try:
            return [float(v) for v in value]
        except (TypeError, ValueError):
            return value

    OTHER_COSTS_7 = [0.75, 120.0]                   # (t, cost) on the validation with cost_fn = 1, cost_fp = 5
    labels_small, decisions_small = np.array([1, 1, 0, 0, 1]), np.array([1, 0, 1, 0, 0])
    costs_ok = False
    try:
        labels_arg, decisions_arg = labels_small.copy(), decisions_small.copy()
        small = total_cost(labels_arg, decisions_arg, 5, 1)
    except NotImplementedError:
        print("⏳ CP3.7 a) : total_cost n'est pas encore écrite.")
    except Exception as exc:
        verdict("CP3.7", False, "", f"total_cost lève une erreur : {type(exc).__name__}: {exc}")
    else:
        if small is None:
            verdict("CP3.7", False, "", "total_cost ne renvoie rien : as-tu oublié le return ?")
        else:
            verdict("CP3.7", isinstance(small, int) and not isinstance(small, bool),
                    "total_cost renvoie un int.",
                    f"total_cost doit renvoyer un int Python (ici : {type(small).__name__}) : return int(...).")
            verdict("CP3.7", np.array_equal(labels_arg, labels_small) and np.array_equal(decisions_arg, decisions_small),
                    "total_cost ne modifie pas ses arguments.",
                    "total_cost modifie y_true ou y_pred (par exemple avec -=) : elle doit seulement les lire.")
            swapped = safe(total_cost, labels_small.copy(), decisions_small.copy(), 1, 5)
            costs_ok = verdict("CP3.7", small == 11 and swapped == 7,
                               "total_cost compte cost_fn par faux négatif et cost_fp par faux positif.",
                               "sur labels [1, 1, 0, 0, 1] et décisions [1, 0, 1, 0, 0] (deux faux négatifs, un faux "
                               f"positif), le coût vaut 2 × 5 + 1 × 1 = 11 avec les coûts (5, 1), et 2 × 1 + 1 × 5 = 7 "
                               f"avec (1, 5) ; ta fonction renvoie {small!r} et {swapped!r}.")
    try:
        proba_copy, labels_copy = PROBA_VAL.copy(), Y_VAL.copy()
        chosen = min_cost_threshold(proba_copy, labels_copy, 5, 1, THRESHOLDS)
    except NotImplementedError:
        print("⏳ CP3.7 b) : min_cost_threshold n'est pas encore écrite.")
    except Exception as exc:
        verdict("CP3.7", False, "", f"min_cost_threshold lève une erreur : {type(exc).__name__}: {exc}")
    else:
        pair = None
        try:
            pair = [float(v) for v in chosen]
        except (TypeError, ValueError):
            pass
        if chosen is None:
            verdict("CP3.7", False, "", "min_cost_threshold ne renvoie rien : as-tu oublié le return ?")
        elif pair is None or len(pair) != 2:
            verdict("CP3.7", False, "", f"min_cost_threshold doit renvoyer un couple (t, coût), pas {chosen!r}"[:300])
        else:
            verdict("CP3.7", np.array_equal(proba_copy, PROBA_VAL) and np.array_equal(labels_copy, Y_VAL),
                    "min_cost_threshold ne modifie pas ses arguments.",
                    "min_cost_threshold modifie proba ou y_true : elle doit seulement les lire.")
            tie = as_pair(safe(min_cost_threshold, np.array([0.2, 0.4, 0.6, 0.8]), np.array([0, 1, 0, 1]), 1, 1,
                               np.array([0.7, 0.5, 0.3])))
            verdict("CP3.7", tie == [0.7, 1.0],
                    "à égalité de coût, le premier seuil de thresholds l'emporte, dans l'ordre reçu.",
                    f"sur proba [0,2 ; 0,4 ; 0,6 ; 0,8], labels [0, 1, 0, 1] et seuils [0,7 ; 0,5 ; 0,3] (dans cet "
                    f"ordre), les trois seuils coûtent 1, 2 et 1 : on attend (0.7, 1), le premier minimum dans l'ordre "
                    f"de thresholds ; ta fonction renvoie {tie!r}.")
            edge = as_pair(safe(min_cost_threshold, np.array([0.5, 0.5]), np.array([1, 1]), 5, 1, np.array([0.5, 0.6])))
            verdict("CP3.7", edge == [0.5, 0.0],
                    "une probabilité égale au seuil donne la classe 1 (≥).",
                    f"sur proba [0,5 ; 0,5], labels [1, 1] et seuils [0,5 ; 0,6], on attend (0.5, 0) : une probabilité "
                    f"égale au seuil donne la classe 1 (proba >= t) ; ta fonction renvoie {edge!r}.")
            other_costs = as_pair(safe(min_cost_threshold, PROBA_VAL.copy(), Y_VAL.copy(), 1, 5, THRESHOLDS))
            verdict("CP3.7", other_costs == OTHER_COSTS_7,
                    "avec d'autres coûts (1, 5), le seuil change comme il doit.",
                    f"avec les coûts cost_fn = 1 et cost_fp = 5 sur la validation, on attend {tuple(OTHER_COSTS_7)} ; "
                    f"ta fonction renvoie {other_costs!r} : utilise-t-elle bien cost_fn et cost_fp ?")
            code_check("CP3.7c", pair, "classe 1 quand proba >= t ; le coût compte 5 par faux négatif et 1 par faux "
                       "positif ; le premier seuil de plus petit coût.")
            if costs_ok:
                test_costs = [safe(total_cost, Y_TEST.copy(), (PROBA_TEST >= t).astype(int), 5, 1) for t in (pair[0], 0.5)]
                code_check("CP3.7e", test_costs, "les coûts du test, avec le seuil de c), puis avec 0,5.")

### Les réponses de ta feuille

**CP3.1 — Prédire la sortie de six extraits scikit-learn (une lettre par extrait)**

In [ ]:
# CP3.1: the values written on your answer sheet
answer_CP3_1a = ...  # a) a letter A, B, C or D
answer_CP3_1b = ...  # b) a letter A, B, C or D
answer_CP3_1c = ...  # c) a letter A, B, C or D
answer_CP3_1d = ...  # d) a letter A, B, C or D
answer_CP3_1e = ...  # e) a letter A, B, C or D
answer_CP3_1f = ...  # f) a letter A, B, C or D

if exam_over():
    for letter, answer in zip("abcdef", [as_letters(answer_CP3_1a), as_letters(answer_CP3_1b), as_letters(answer_CP3_1c), as_letters(answer_CP3_1d), as_letters(answer_CP3_1e), as_letters(answer_CP3_1f)]):
        RESULTS[f"CP3.1{letter}"] = wb.check(f"CP3.1{letter}", answer)

**CP3.2 — Standardiser et encoder un mini-tableau**

In [ ]:
# CP3.2: the values written on your answer sheet
answer_CP3_2a = ...  # a) [mean, std], 4 decimals
answer_CP3_2b = ...  # b) z-score of penguin 2, 3 decimals
answer_CP3_2c = ...  # c) z-score of T1, 3 decimals
answer_CP3_2d = ...  # d) min-max of T1, 3 decimals
answer_CP3_2e = ...  # e) one-hot of penguin 3 (a list)
answer_CP3_2f = ...  # f) one-hot of T2 (a list)
answer_CP3_2g = ...  # g) row of T2 from the ColumnTransformer (a list, 3 decimals)

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP3_2a, answer_CP3_2b, answer_CP3_2c, answer_CP3_2d, answer_CP3_2e, answer_CP3_2f, answer_CP3_2g]):
        RESULTS[f"CP3.2{letter}"] = wb.check(f"CP3.2{letter}", answer)

**CP3.3 — Gini et meilleur seuil**

In [ ]:
# CP3.3: the values written on your answer sheet
answer_CP3_3a = ...  # a) Gini of the root
answer_CP3_3b = ...  # b) number of candidate thresholds
answer_CP3_3c = ...  # c) weighted Gini of s = 4, 3 decimals
answer_CP3_3d = ...  # d) the best threshold
answer_CP3_3e = ...  # e) [weighted Gini, gain], 3 decimals
answer_CP3_3f = ...  # f) entropy of the right child in bits, 3 decimals

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP3_3a, answer_CP3_3b, answer_CP3_3c, answer_CP3_3d, answer_CP3_3e, answer_CP3_3f]):
        RESULTS[f"CP3.3{letter}"] = wb.check(f"CP3.3{letter}", answer)

**CP3.4 — kNN avant et après standardisation**

In [ ]:
# CP3.4: the values written on your answer sheet
answer_CP3_4a = ...  # a) the 3 nearest neighbours on the raw measures (a list)
answer_CP3_4b = ...  # b) the species predicted on the raw measures
answer_CP3_4c = ...  # c) [z_mass, z_bill] of Q, 3 decimals
answer_CP3_4d = ...  # d) the 3 nearest neighbours after standardization (a list)
answer_CP3_4e = ...  # e) the species predicted after standardization

if exam_over():
    for letter, answer in zip("abcde", [answer_CP3_4a, answer_CP3_4b, answer_CP3_4c, answer_CP3_4d, answer_CP3_4e]):
        RESULTS[f"CP3.4{letter}"] = wb.check(f"CP3.4{letter}", answer)

**CP3.5 — AdaBoost et gradient boosting**

In [ ]:
# CP3.5: the values written on your answer sheet
answer_CP3_5a = ...  # a) weighted error
answer_CP3_5b = ...  # b) alpha, 3 decimals
answer_CP3_5c = ...  # c) normalized weight of a misclassified example, 4 decimals
answer_CP3_5d = ...  # d) normalized weight of a well-classified example, 4 decimals
answer_CP3_5e = ...  # e) alpha with K = 4 classes, 3 decimals
answer_CP3_5f = ...  # f) F0
answer_CP3_5g = ...  # g) residuals (a list)
answer_CP3_5h = ...  # h) [s, left value, right value]
answer_CP3_5i = ...  # i) F1(1), 2 decimals
answer_CP3_5j = ...  # j) training MSE after the step, 4 decimals

if exam_over():
    for letter, answer in zip("abcdefghij", [answer_CP3_5a, answer_CP3_5b, answer_CP3_5c, answer_CP3_5d, answer_CP3_5e, answer_CP3_5f, answer_CP3_5g, answer_CP3_5h, answer_CP3_5i, answer_CP3_5j]):
        RESULTS[f"CP3.5{letter}"] = wb.check(f"CP3.5{letter}", answer)

**CP3.6 — Les lettres des trois fuites (a)**

In [ ]:
# CP3.6: the values written on your answer sheet
answer_CP3_6a = ...  # a) three letters, for example "XYZ"

if exam_over():
    for letter, answer in zip("a", [as_letters(answer_CP3_6a)]):
        RESULTS[f"CP3.6{letter}"] = wb.check(f"CP3.6{letter}", answer)

**CP3.7 — Le seuil théorique (d)**

In [ ]:
# CP3.7: the values written on your answer sheet
answer_CP3_7d = ...  # d) the theoretical threshold, 4 decimals

if exam_over():
    for letter, answer in zip("d", [answer_CP3_7d]):
        RESULTS[f"CP3.7{letter}"] = wb.check(f"CP3.7{letter}", answer)

**CP3.8 — Lire les courbes de la figure**

In [ ]:
# CP3.8: the values written on your answer sheet
answer_CP3_8a = ...  # a) the number of the training curve (1 or 2)
answer_CP3_8b = ...  # b) a letter A, B, C or D
answer_CP3_8c = ...  # c) a letter A, B or C
answer_CP3_8d = ...  # d) a letter A, B or C
answer_CP3_8e = ...  # e) the learning rate (1 or 0.1)
answer_CP3_8f = ...  # f) a letter A, B, C or D
answer_CP3_8g = ...  # g) True / False (ou "vrai" / "faux")

if exam_over():
    for letter, answer in zip("abcdefg", [as_curve(answer_CP3_8a), as_letters(answer_CP3_8b), as_letters(answer_CP3_8c), as_letters(answer_CP3_8d), answer_CP3_8e, as_letters(answer_CP3_8f), answer_CP3_8g]):
        RESULTS[f"CP3.8{letter}"] = wb.check(f"CP3.8{letter}", answer)

**CP3.11 — Parties I et II**

In [ ]:
# CP3.11: the values written on your answer sheet
answer_CP3_11a = ...  # a) [TP, FP, FN, TN]
answer_CP3_11b = ...  # b) precision, 3 decimals
answer_CP3_11c = ...  # c) recall, 3 decimals
answer_CP3_11d = ...  # d) F1, 3 decimals
answer_CP3_11e = ...  # e) accuracy, 3 decimals
answer_CP3_11f = ...  # f) [validation fold size, class 1 in it]
answer_CP3_11g = ...  # g) [training size, class 1 in it]
answer_CP3_11h = ...  # h) number of models

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP3_11a, answer_CP3_11b, answer_CP3_11c, answer_CP3_11d, answer_CP3_11e, answer_CP3_11f, answer_CP3_11g, answer_CP3_11h]):
        RESULTS[f"CP3.11{letter}"] = wb.check(f"CP3.11{letter}", answer)

**CP3.12 — Vrai ou faux justifiés (le verdict seulement ; les justifications se notent avec le corrigé)**

In [ ]:
# CP3.12: the values written on your answer sheet
answer_CP3_12a = ...  # a) True / False (ou "vrai" / "faux")
answer_CP3_12b = ...  # b) True / False (ou "vrai" / "faux")
answer_CP3_12c = ...  # c) True / False (ou "vrai" / "faux")
answer_CP3_12d = ...  # d) True / False (ou "vrai" / "faux")
answer_CP3_12e = ...  # e) True / False (ou "vrai" / "faux")
answer_CP3_12f = ...  # f) True / False (ou "vrai" / "faux")

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP3_12a, answer_CP3_12b, answer_CP3_12c, answer_CP3_12d, answer_CP3_12e, answer_CP3_12f]):
        RESULTS[f"CP3.12{letter}"] = wb.check(f"CP3.12{letter}", answer)

**CP3.13 — Variance d'une moyenne de modèles corrélés**

In [ ]:
# CP3.13: the values written on your answer sheet
answer_CP3_13b = ...  # b) variance for M = 10, 3 decimals
answer_CP3_13c = ...  # c) the limit for M -> infinity
answer_CP3_13d = ...  # d) the smallest M
answer_CP3_13e = ...  # e) [limit, variance for M = 30], 3 decimals

if exam_over():
    for letter, answer in zip("bcde", [answer_CP3_13b, answer_CP3_13c, answer_CP3_13d, answer_CP3_13e]):
        RESULTS[f"CP3.13{letter}"] = wb.check(f"CP3.13{letter}", answer)

**CP3.14 — Coût d'une recherche en grille**

In [ ]:
# CP3.14: the values written on your answer sheet
answer_CP3_14a = ...  # a) number of trainings
answer_CP3_14b = ...  # b) duration in minutes
answer_CP3_14c = ...  # c) number of trainings of the nested cross-validation
answer_CP3_14d = ...  # d) number of trainings of the random search

if exam_over():
    for letter, answer in zip("abcd", [answer_CP3_14a, answer_CP3_14b, answer_CP3_14c, answer_CP3_14d]):
        RESULTS[f"CP3.14{letter}"] = wb.check(f"CP3.14{letter}", answer)

### Bilan

In [ ]:
# Summary of the automatic checks
import re


def natural(sub_id):
    """CP3.2a before CP3.13a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP3\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    checked = [ok for props in PROPERTIES.values() for _, ok in props]
    print(f"Propriétés du code (CP3.6, CP3.7) : {sum(checked)} sur {len(checked)}.")
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; les explications et les démonstrations (CP3.2 h, CP3.4 f, CP3.5 k, CP3.6 b et e, CP3.7 d (la démonstration) et f, CP3.9, CP3.10, CP3.12 (les justifications), CP3.13 a et f) ; et, pour CP3.6 et CP3.7, les nombres recopiés sur ta feuille (seules tes fonctions sont vérifiées). Tout cela se corrige avec `03_examen_corrige.md`. Ensuite : la remédiation du corrigé (relis la synthèse, `05_synthese.md`, pour les notions ratées), puis le mini-projet.